# TrashNet completo: transformaciones controladas para VGG-16

Este cuaderno prepara las cuatro transformaciones (brillo, contraste, rotación y zoom), cada una en tres niveles, usando **todas las imágenes disponibles por clase**. No balancea ni entrena un clasificador: cada imagen será comparada posteriormente con su propia versión transformada.

Las imágenes transformadas se guardan en PNG para no añadir compresión JPEG como factor de confusión.

In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


## 1. Configuración
Modifica únicamente `INPUT_ROOT` si el TrashNet completo se encuentra en otra carpeta. La carpeta debe contener las seis subcarpetas de clase.

In [2]:
from pathlib import Path

BASE_DIR = Path('/content/gdrive/MyDrive/2026-0')
INPUT_ROOT = BASE_DIR / 'DatasetOriginalCompleto'
OUTPUT_ROOT = BASE_DIR / 'TrashNet_transformaciones_completo_png'
MANIFEST_PATH = BASE_DIR / 'manifest_trashnet_completo.csv'

CLASSES = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
EXPECTED_COUNTS = {
    'cardboard': 403,
    'glass': 501,
    'metal': 410,
    'paper': 594,
    'plastic': 482,
    'trash': 137,
}
VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp'}
OVERWRITE = False

BRIGHTNESS = {'low': 1.2, 'medium': 1.4, 'high': 1.6}
CONTRAST = {'low': 1.2, 'medium': 1.4, 'high': 1.6}
ROTATION = {'low': 10, 'medium': 30, 'high': 60}
ZOOM = {'low': 1.10, 'medium': 1.25, 'high': 1.50}

print('Entrada:', INPUT_ROOT)
print('Salida:', OUTPUT_ROOT)

Entrada: /content/gdrive/MyDrive/2026-0/DatasetOriginalCompleto
Salida: /content/gdrive/MyDrive/2026-0/TrashNet_transformaciones_completo_png


## 2. Verificación del dataset completo
La ejecución se detiene si falta una clase o si los conteos no coinciden con TrashNet. Así evitamos procesar accidentalmente el subconjunto balanceado de 822 imágenes.

In [3]:
def list_images(class_dir):
    return sorted(
        [p for p in class_dir.iterdir() if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS],
        key=lambda p: p.name.lower()
    )

if not INPUT_ROOT.exists():
    raise FileNotFoundError(f'No existe la carpeta de entrada: {INPUT_ROOT}')

files_by_class = {}
for class_name in CLASSES:
    class_dir = INPUT_ROOT / class_name
    if not class_dir.exists():
        raise FileNotFoundError(f'Falta la clase: {class_dir}')
    files_by_class[class_name] = list_images(class_dir)

observed_counts = {c: len(files_by_class[c]) for c in CLASSES}
for c in CLASSES:
    status = 'OK' if observed_counts[c] == EXPECTED_COUNTS[c] else 'REVISAR'
    print(f'{c:10s}: {observed_counts[c]:4d} | esperado: {EXPECTED_COUNTS[c]:4d} | {status}')

total = sum(observed_counts.values())
print(f'\nTotal observado: {total} | esperado: {sum(EXPECTED_COUNTS.values())}')

if observed_counts != EXPECTED_COUNTS:
    raise ValueError(
        'Los conteos no corresponden al TrashNet completo. '        'Revisa INPUT_ROOT antes de continuar.'
    )

cardboard :  403 | esperado:  403 | OK
glass     :  501 | esperado:  501 | OK
metal     :  410 | esperado:  410 | OK
paper     :  594 | esperado:  594 | OK
plastic   :  482 | esperado:  482 | OK
trash     :  137 | esperado:  137 | OK

Total observado: 2527 | esperado: 2527


## 3. Funciones de transformación y guardado
El zoom conserva el procedimiento del estudio: recorte central proporcional y reescalado bilineal al tamaño original.

In [4]:
from PIL import Image
from torchvision.transforms import functional as F
from torchvision.transforms import InterpolationMode

def zoom_image(img, zoom_factor):
    width, height = img.size
    crop_width = max(1, int(width / zoom_factor))
    crop_height = max(1, int(height / zoom_factor))
    left = (width - crop_width) // 2
    top = (height - crop_height) // 2
    crop = img.crop((left, top, left + crop_width, top + crop_height))
    return crop.resize((width, height), resample=Image.Resampling.BILINEAR)

def apply_transformation(img, transformation, value):
    if transformation == 'brightness':
        return F.adjust_brightness(img, brightness_factor=value)
    if transformation == 'contrast':
        return F.adjust_contrast(img, contrast_factor=value)
    if transformation == 'rotation':
        return F.rotate(
            img, angle=value, interpolation=InterpolationMode.BILINEAR,
            expand=False, fill=0
        )
    if transformation == 'zoom':
        return zoom_image(img, value)
    raise ValueError(f'Transformación no reconocida: {transformation}')

TRANSFORMATIONS = {
    'brightness': BRIGHTNESS,
    'contrast': CONTRAST,
    'rotation': ROTATION,
    'zoom': ZOOM,
}

## 4. Generación de las 12 condiciones
Se producirán 30 324 imágenes transformadas (2527 imágenes × 12 condiciones). `OVERWRITE=False` permite reanudar la ejecución sin volver a escribir archivos ya existentes.

In [5]:
import csv
from tqdm.auto import tqdm

manifest_rows = []
written = 0
skipped = 0
errors = []

total_outputs = total * sum(len(levels) for levels in TRANSFORMATIONS.values())
with tqdm(total=total_outputs, desc='Transformaciones') as progress:
    for class_name in CLASSES:
        for image_path in files_by_class[class_name]:
            try:
                with Image.open(image_path) as source:
                    img = source.convert('RGB')
                    original_width, original_height = img.size

                    for transformation, levels in TRANSFORMATIONS.items():
                        for level, value in levels.items():
                            output_dir = OUTPUT_ROOT / f'{transformation}_{level}' / class_name
                            output_dir.mkdir(parents=True, exist_ok=True)
                            output_path = output_dir / f'{image_path.stem}.png'

                            if OVERWRITE or not output_path.exists():
                                transformed = apply_transformation(img, transformation, value)
                                transformed.save(output_path, format='PNG', compress_level=4)
                                written += 1
                            else:
                                skipped += 1

                            manifest_rows.append({
                                'class': class_name,
                                'original_filename': image_path.name,
                                'original_path': str(image_path),
                                'transformation': transformation,
                                'level': level,
                                'value': value,
                                'output_filename': output_path.name,
                                'output_path': str(output_path),
                                'original_width': original_width,
                                'original_height': original_height,
                            })
                            progress.update(1)
            except Exception as exc:
                errors.append((str(image_path), repr(exc)))
                progress.update(12)

fieldnames = list(manifest_rows[0].keys())
with MANIFEST_PATH.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(manifest_rows)

print(f'Archivos escritos: {written}')
print(f'Archivos omitidos porque ya existían: {skipped}')
print(f'Errores: {len(errors)}')
print(f'Manifiesto: {MANIFEST_PATH}')

if errors:
    print('Primeros errores:', errors[:10])
    raise RuntimeError('Hubo imágenes que no pudieron procesarse.')

Transformaciones:   0%|          | 0/30324 [00:00<?, ?it/s]

Archivos escritos: 30324
Archivos omitidos porque ya existían: 0
Errores: 0
Manifiesto: /content/gdrive/MyDrive/2026-0/manifest_trashnet_completo.csv


## 5. Auditoría final
Comprueba que cada condición contenga exactamente las mismas imágenes que el dataset original.

In [6]:
audit_rows = []
all_ok = True

for transformation, levels in TRANSFORMATIONS.items():
    for level in levels:
        for class_name in CLASSES:
            output_dir = OUTPUT_ROOT / f'{transformation}_{level}' / class_name
            count = len(list(output_dir.glob('*.png')))
            expected = observed_counts[class_name]
            ok = count == expected
            all_ok &= ok
            audit_rows.append((transformation, level, class_name, count, expected, ok))

for row in audit_rows:
    transformation, level, class_name, count, expected, ok = row
    print(f'{transformation:10s} {level:6s} {class_name:10s}: {count:4d}/{expected:4d}  {"OK" if ok else "REVISAR"}')

print('\nAUDITORÍA FINAL:', 'CORRECTA' if all_ok else 'CON ERRORES')
if not all_ok:
    raise ValueError('Alguna condición no contiene el número esperado de imágenes.')

brightness low    cardboard :  403/ 403  OK
brightness low    glass     :  501/ 501  OK
brightness low    metal     :  410/ 410  OK
brightness low    paper     :  594/ 594  OK
brightness low    plastic   :  482/ 482  OK
brightness low    trash     :  137/ 137  OK
brightness medium cardboard :  403/ 403  OK
brightness medium glass     :  501/ 501  OK
brightness medium metal     :  410/ 410  OK
brightness medium paper     :  594/ 594  OK
brightness medium plastic   :  482/ 482  OK
brightness medium trash     :  137/ 137  OK
brightness high   cardboard :  403/ 403  OK
brightness high   glass     :  501/ 501  OK
brightness high   metal     :  410/ 410  OK
brightness high   paper     :  594/ 594  OK
brightness high   plastic   :  482/ 482  OK
brightness high   trash     :  137/ 137  OK
contrast   low    cardboard :  403/ 403  OK
contrast   low    glass     :  501/ 501  OK
contrast   low    metal     :  410/ 410  OK
contrast   low    paper     :  594/ 594  OK
contrast   low    plastic   :  4

## Siguiente cuaderno
La siguiente etapa será extraer activaciones de Block1–Block5 con VGG-16, comparar cada PNG transformado con su imagen original y exportar un CSV largo con clase, imagen, transformación, intensidad, bloque, similitud coseno y distancia euclidiana relativa.